# LeetCode 625 — Minimum Factorization bằng Python

Link đề: https://leetcode.com/problems/minimum-factorization/ · Độ khó: **Medium** · Chủ đề: Greedy, Math, Prime Factorization

> **Lưu ý (bài Premium 🔒):** đề dưới đây được viết theo hiểu biết về bài này (đối chiếu với bản tóm tắt công khai). Nếu bạn có tài khoản Premium, hãy **đối chiếu lại đề gốc** trên LeetCode, đặc biệt phần ràng buộc và tên hàm.

Skeleton LeetCode:

```python
class Solution:
    def smallestFactorization(self, num: int) -> int:
        ...
```

Ghi chú: bài có **đáp án duy nhất** (một số nguyên, hoặc `0` nếu không có/không vừa 32-bit), nên so sánh bằng `==`. Notebook kiểm chứng bằng nhiều **oracle độc lập**: vét cạn theo bảng "tích các chữ số", và quy hoạch động trên các ước số.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm số nguyên dương **nhỏ nhất** `x` sao cho **tích các chữ số của `x` bằng `num`**; nếu không có hoặc `x` không vừa kiểu số nguyên có dấu 32-bit thì trả `0`.
2. Nắm các công cụ Python: `%` và `//` để bóc chữ số, `str`/`int`, `sorted`, `Counter`, chia hết trong vòng `while`, ngưỡng `2**31 - 1`, và test bằng `functools.lru_cache`.
3. Đi từ **thử `x = 1, 2, 3, ...`** → **liệt kê mọi cách tách thành chữ số (quay lui)** → **tham lam chia cho 9, 8, ..., 2** → **đếm số mũ của các thừa số nguyên tố 2, 3, 5, 7 rồi ghép chữ số**.
4. Hiểu mấu chốt: dùng **ít chữ số nhất** (số ít chữ số thì nhỏ hơn), trong đó chữ số **lớn** "ăn" được nhiều thừa số nhất; sau đó xếp các chữ số **tăng dần** để số nhỏ nhất.
5. Tránh các lỗi kinh điển: duyệt chữ số từ 2 lên 9, ghép chữ số sai thứ tự, quên kiểm tra số dư còn lại `> 1`, quên trường hợp `num = 1`, quên giới hạn 32-bit.

## 2. Đề bài & ví dụ

Cho một số nguyên dương `num`. Trả về **số nguyên dương nhỏ nhất `x`** mà **tích các chữ số** của `x` bằng `num`. Nếu **không tồn tại** `x` như vậy, hoặc `x` **không vừa** trong số nguyên có dấu **32-bit** (`x > 2^31 - 1`), trả về `0`.

**Ví dụ 1:**

```
Input:  num = 48
Output: 68
```

`6 × 8 = 48`, và `68` là số nhỏ nhất thoả điều kiện (các số khác như `86`, `238`, `246`... đều lớn hơn).

**Ví dụ 2:**

```
Input:  num = 15
Output: 35
```

`3 × 5 = 15`.

**Ràng buộc:** `1 <= num <= 2^31 - 1`.

**Hai điều cần để ý:**

- `num = 1`: số nhỏ nhất có tích chữ số bằng `1` là chính `1` (một chữ số `1`), nên đáp án là `1`.
- `num` có ước nguyên tố lớn hơn `7` (như `11`, `13`, `22`) thì **không** tách được thành tích các chữ số nên đáp án `0`.

In [1]:
import sys
import time
import random
from collections import Counter
from functools import lru_cache
from itertools import product

INT_MAX = 2 ** 31 - 1
print("INT_MAX =", INT_MAX)
for num, want in [(48, 68), (15, 35)]:
    print(f"num = {num}: 6x8 = {6 * 8}" if num == 48 else f"num = {num}: 3x5 = {3 * 5}", "-> kỳ vọng", want)

INT_MAX = 2147483647
num = 48: 6x8 = 48 -> kỳ vọng 68
num = 15: 3x5 = 15 -> kỳ vọng 35


## 3. Kiến thức nền

### 3.1 Bóc chữ số bằng `%` và `//`

`n % 10` là **chữ số hàng đơn vị**, `n // 10` là số sau khi bỏ chữ số đó. Lặp lại tới khi `n == 0` thì lấy hết chữ số. Ta dùng để tính **tích các chữ số** của một số `x`.

In [2]:
def digit_product(x):
    p = 1
    while x:
        p *= x % 10
        x //= 10
    return p


for x in (68, 35, 238, 101, 5):
    print(f"digit_product({x}) = {digit_product(x)}")
print("bóc từng chữ số của 3820:", [(3820 // 10 ** k) % 10 for k in range(3, -1, -1)])

digit_product(68) = 48
digit_product(35) = 15
digit_product(238) = 48
digit_product(101) = 0
digit_product(5) = 5
bóc từng chữ số của 3820: [3, 8, 2, 0]


### 3.2 Ghép chữ số thành số

Từ danh sách chữ số `[3, 5]` muốn được `35`. Có hai cách: ghép chuỗi `int("".join(map(str, ds)))` hoặc dùng công thức Horner `ans = ans * 10 + d` duyệt từ chữ số cao xuống thấp. Trong lời giải chuẩn ta chèn từ chữ số **thấp** lên: `ans = mul * d + ans; mul *= 10` (đặt `d` vào hàng `mul`).

In [3]:
ds = [3, 5]
print(int("".join(map(str, ds))))

ans = 0
for d in ds:                    # Horner: duyệt từ chữ số cao
    ans = ans * 10 + d
print(ans)

ans, mul = 0, 1
for d in reversed(ds):          # chèn từ chữ số thấp lên: mul = 1, 10, 100, ...
    ans = mul * d + ans
    mul *= 10
print(ans)

35
35
35


### 3.3 Chia hết và vòng `while`

`num % i == 0` nghĩa là `i` là ước của `num`. `while num % i == 0: num //= i` rút hết mọi thừa số `i` ra khỏi `num` (đếm xem rút được bao nhiêu lần). Ví dụ `72 = 8 × 9`: rút `9` một lần được `8`, rút `8` một lần được `1`.

In [4]:
num = 72
for i in (9, 8, 7):
    k = 0
    while num % i == 0:
        num //= i
        k += 1
    print(f"rút {k} lần thừa số {i}, còn lại num = {num}")

rút 1 lần thừa số 9, còn lại num = 8
rút 1 lần thừa số 8, còn lại num = 1
rút 0 lần thừa số 7, còn lại num = 1


### 3.4 Vì sao chỉ dùng chữ số `2..9`

- Chữ số `0` làm tích bằng `0` — vô dụng vì `num >= 1`.
- Chữ số `1` không đổi tích nhưng **làm số dài thêm** (nhiều chữ số hơn thì lớn hơn), nên chỉ dùng khi `num = 1` (khi đó đáp án là `1`).
- Vậy cần viết `num` thành **tích các chữ số trong `{2, ..., 9}`**. Mỗi chữ số có phân tích nguyên tố: `2, 3, 5, 7` là nguyên tố; `4 = 2²`, `6 = 2·3`, `8 = 2³`, `9 = 3²`. Suy ra `num` chỉ được chứa các thừa số nguyên tố `2, 3, 5, 7`. Nếu có thừa số nguyên tố lớn hơn (`11, 13, ...`) thì không có đáp án.

In [5]:
def small_prime_part(n):
    """Rút hết thừa số 2,3,5,7; trả (số dư, số mũ của 2,3,5,7)."""
    exps = []
    for p in (2, 3, 5, 7):
        e = 0
        while n % p == 0:
            n //= p
            e += 1
        exps.append(e)
    return n, exps


for n in (48, 15, 13, 22, 360, 1):
    rest, e = small_prime_part(n)
    print(f"{n:4d}: mũ (2,3,5,7) = {e}, số dư = {rest} ->", "tách được" if rest == 1 else "KHÔNG tách được (có ước nguyên tố > 7)")

  48: mũ (2,3,5,7) = [4, 1, 0, 0], số dư = 1 -> tách được
  15: mũ (2,3,5,7) = [0, 1, 1, 0], số dư = 1 -> tách được
  13: mũ (2,3,5,7) = [0, 0, 0, 0], số dư = 13 -> KHÔNG tách được (có ước nguyên tố > 7)
  22: mũ (2,3,5,7) = [1, 0, 0, 0], số dư = 11 -> KHÔNG tách được (có ước nguyên tố > 7)
 360: mũ (2,3,5,7) = [3, 2, 1, 0], số dư = 1 -> tách được
   1: mũ (2,3,5,7) = [0, 0, 0, 0], số dư = 1 -> tách được


### 3.5 Số ít chữ số thì nhỏ hơn; cùng số chữ số thì so từ trái sang

Hai số nguyên dương: số có **nhiều chữ số hơn** thì lớn hơn (ví dụ `99 < 100`). Nếu cùng số chữ số thì so **từ chữ số cao nhất**. Cho một **tập (đa tập) chữ số**, số nhỏ nhất ghép từ nó là **sắp xếp chữ số tăng dần** (ví dụ `{8, 3, 5} → 358`).

In [6]:
ds = [8, 3, 5]
print("tăng dần :", int("".join(map(str, sorted(ds)))))
print("giảm dần :", int("".join(map(str, sorted(ds, reverse=True)))))
print("99 < 100:", 99 < 100, "| 238 < 68 ?", 238 < 68, "(68 có ít chữ số hơn nên nhỏ hơn)")

tăng dần : 358
giảm dần : 853
99 < 100: True | 238 < 68 ? False (68 có ít chữ số hơn nên nhỏ hơn)


### 3.6 Giới hạn 32-bit

Số nguyên có dấu 32-bit nhỏ hơn hoặc bằng `2**31 - 1 = 2147483647` (10 chữ số). Trong Python số nguyên không tràn, nên ta phải **tự so sánh** với `2**31 - 1` ở cuối để đổi thành `0`. (Trong Java/C++ phải dùng `long` để chứa kết quả trung gian rồi mới so sánh.)

In [7]:
print(INT_MAX, len(str(INT_MAX)), "chữ số")
print(5 ** 13, "<= INT_MAX ?", 5 ** 13 <= INT_MAX, "| đáp án sẽ là 13 chữ số 5 =", int("5" * 13), "> INT_MAX")

2147483647 10 chữ số
1220703125 <= INT_MAX ? True | đáp án sẽ là 13 chữ số 5 = 5555555555555 > INT_MAX


## 4. Ý tưởng nền tảng / trực giác

Tính tay `num = 48`. Các cách viết `48` thành tích các chữ số `2..9` (bỏ qua thứ tự), kèm số nhỏ nhất ghép được:

| cách tách | số chữ số | số nhỏ nhất |
|---|---|---|
| `6 × 8` | 2 | `68` |
| `2 × 3 × 8` | 3 | `238` |
| `2 × 4 × 6` | 3 | `246` |
| `3 × 4 × 4` | 3 | `344` |
| `2 × 2 × 2 × 6` | 4 | `2226` |
| `2 × 2 × 3 × 4` | 4 | `2234` |
| `2 × 2 × 2 × 2 × 3` | 5 | `22223` |

Quy tắc đọc ra: (1) **ít chữ số nhất** thắng (`68` có 2 chữ số); (2) muốn ít chữ số thì chữ số phải **lớn**; (3) xếp chữ số tăng dần cho ra số nhỏ nhất.

Cách tạo chữ số lớn: chia `48` cho chữ số lớn nhất có thể: `9`? không chia hết. `8`? có: `48 / 8 = 6` → còn `6` → chữ số `6`. Hai chữ số `{8, 6}` → `68`. Đó chính là thuật toán tham lam: **thử chia cho 9, 8, 7, ..., 2**.

In [8]:
# Liệt kê mọi cách tách 48 thành tích các chữ số 2..9 (đa tập, không phân biệt thứ tự)
def all_factorizations(n, start=2):
    """Sinh mọi dãy chữ số không giảm (>= start) có tích n."""
    if n == 1:
        yield []
        return
    for d in range(start, 10):
        if n % d == 0:
            for rest in all_factorizations(n // d, d):
                yield [d] + rest


rows = sorted(all_factorizations(48), key=lambda ds: (len(ds), ds))
for ds in rows:
    print(f"{' x '.join(map(str, ds)):20s} {len(ds)} chữ số -> {int(''.join(map(str, ds)))}")
print("nhỏ nhất:", min(int("".join(map(str, ds))) for ds in rows))

6 x 8                2 chữ số -> 68
2 x 3 x 8            3 chữ số -> 238
2 x 4 x 6            3 chữ số -> 246
3 x 4 x 4            3 chữ số -> 344
2 x 2 x 2 x 6        4 chữ số -> 2226
2 x 2 x 3 x 4        4 chữ số -> 2234
2 x 2 x 2 x 2 x 3    5 chữ số -> 22223
nhỏ nhất: 68


## 5. Cách 1 — Brute Force: thử `x = 1, 2, 3, ...`

**Ý tưởng:** đúng theo đề. Duyệt `x` tăng dần từ `1`; số đầu tiên có `digit_product(x) == num` là đáp án vì ta đi từ nhỏ đến lớn. Chỉ có hiệu lực nếu có chặn trên (nếu không có đáp án, vòng lặp không bao giờ dừng!). Ta thêm tham số `limit` để thử trong phạm vi nhỏ.

In [9]:
def smallest_brute(num, limit=10 ** 6):
    """Thử x = 1..limit; trả x đầu tiên có tích chữ số = num, hoặc 0 nếu không thấy trong limit."""
    for x in range(1, limit + 1):
        if digit_product(x) == num:
            return x
    return 0


for num in (48, 15, 1, 7, 13, 100):
    print(f"num={num:3d} -> {smallest_brute(num)}")
assert smallest_brute(48) == 68 and smallest_brute(15) == 35 and smallest_brute(1) == 1

num= 48 -> 68
num= 15 -> 35
num=  1 -> 1
num=  7 -> 7


num= 13 -> 0
num=100 -> 455


**Độ phức tạp:** thời gian `O(đáp án · số chữ số)`; với `num` lớn đáp án có thể là số 9 chữ số (`~10^9` lần thử) và nếu **không có đáp án** (như `num = 13`) ta phải duyệt tới hết `limit` rồi mới biết. Bộ nhớ `O(1)`. Không dùng được trên ràng buộc `num ≤ 2^31 − 1`.

**Nút thắt:** ta đang đoán mò đáp án rồi kiểm tra, thay vì **xây** đáp án từ các ước của `num`. Ý tưởng tiếp theo: đi từ `num` và tách nó ra.

## 6. Ý tưởng thuật toán / bước đột phá

Đảo ngược bài toán: thay vì đoán `x` rồi tính tích chữ số, hãy **tách `num` thành các chữ số**. Một đa tập chữ số `{d1, ..., dk}` có tích `num` cho ra số nhỏ nhất bằng cách xếp tăng dần. Ta cần chọn đa tập sao cho số kết quả nhỏ nhất, tức là:

1. **Số chữ số `k` nhỏ nhất** (ưu tiên số 1).
2. Với cùng `k`, đa tập **nhỏ nhất theo thứ tự từ điển** sau khi sắp xếp tăng dần.

Lý do chia cho chữ số **lớn trước**: một chữ số lớn "ăn" được nhiều thừa số nhất nên giúp `k` nhỏ. Với tham lam `9, 8, 7, ..., 2`:

- `9 = 3²` và `8 = 2³` nuốt nhiều thừa số nguyên tố nhất; `7`, `5` là nguyên tố nên **bắt buộc** là chữ số; `6 = 2·3`, `4 = 2²` xử lý phần dư còn lại.
- Hai chữ số nào có tích `≤ 9` luôn có thể **gộp thành một chữ số** (ít chữ số hơn → số nhỏ hơn). Cho nên trong đáp án tối ưu mọi cặp chữ số có tích `≥ 10`.

Ta sẽ chứng minh gọn ở mục 10 và kiểm chứng bằng **oracle độc lập** (vét cạn theo bảng tích chữ số, quy hoạch động theo ước số) trên mọi `num ≤ 20000` ở mục 12.

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta xét thêm 3 cách: **Cách 2** quay lui liệt kê mọi cách tách, **Cách 3** tham lam chia cho `9..2` (chuẩn), **Cách 4** đếm số mũ nguyên tố rồi ghép chữ số.

### Cách 2 — Quay lui: liệt kê mọi đa tập chữ số có tích `num`

**Giải thích từng đoạn:**

1. Hàm `dfs(rem, start, digits)`: còn phải tách `rem`, chữ số tiếp theo `>= start` (để mỗi đa tập chỉ sinh **một lần**, theo thứ tự không giảm), `digits` là các chữ số đã chọn.
2. Nếu `rem == 1`: đã tách xong → ghép `digits` (đã không giảm nên đã là số nhỏ nhất của đa tập này) thành số, cập nhật số nhỏ nhất.
3. Ngược lại với mỗi chữ số `d` từ `start` đến `9` mà `d | rem`: chọn `d`, gọi `dfs(rem // d, d, digits + [d])`.
4. Trường hợp riêng: `num = 1` → đáp án `1`.

In [10]:
def smallest_backtrack(num):
    if num == 1:
        return 1
    best = [None]

    def dfs(rem, start, digits):
        if rem == 1:
            value = int("".join(map(str, digits)))
            if best[0] is None or value < best[0]:
                best[0] = value
            return
        for d in range(start, 10):
            if rem % d == 0:
                dfs(rem // d, d, digits + [d])

    dfs(num, 2, [])
    return best[0] if best[0] is not None and best[0] <= INT_MAX else 0


for num in (48, 15, 1, 13, 100, 360):
    print(f"num={num:4d} -> {smallest_backtrack(num)}")
assert smallest_backtrack(48) == 68 and smallest_backtrack(15) == 35 and smallest_backtrack(13) == 0

num=  48 -> 68
num=  15 -> 35
num=   1 -> 1
num=  13 -> 0
num= 100 -> 455
num= 360 -> 589


In [11]:
# Code nộp LeetCode cho Cách 2
class SolutionBacktrack:
    def smallestFactorization(self, num: int) -> int:
        if num == 1:
            return 1
        best = []

        def dfs(rem: int, start: int, digits: list) -> None:
            if rem == 1:
                best.append(int("".join(map(str, digits))))
                return
            for d in range(start, 10):
                if rem % d == 0:
                    dfs(rem // d, d, digits + [d])

        dfs(num, 2, [])
        if not best:
            return 0
        ans = min(best)
        return ans if ans <= 2 ** 31 - 1 else 0

**Độ phức tạp:** bằng số cách tách `num` thành đa tập chữ số `2..9` (nhỏ: `num ≤ 2^31` chỉ chứa `2,3,5,7` với số mũ nhỏ, số cách cỡ vài nghìn). Đủ nhanh cho bài, nhưng `rem` không phải bội của chữ số nào thì cụt ngay.

Có thể đếm số cách tách và độ sâu để thấy quy mô:

In [12]:
def count_all(num):
    return sum(1 for _ in all_factorizations(num))


for num in (48, 360, 2 ** 20, 2 ** 15 * 3 ** 8 * 5 * 7):
    print(f"num={num:>12d}: {count_all(num):6d} cách tách")

num=          48:      7 cách tách
num=         360:      9 cách tách
num=     1048576:     44 cách tách
num=  7524679680:    490 cách tách


### Cách 3 — Tham lam: chia cho `9, 8, ..., 2` (cách chuẩn)

**Giải thích từng đoạn:**

1. `num < 2` (tức `num = 1`) → trả luôn `num` (`= 1`).
2. Duyệt chữ số `i` từ `9` xuống `2`. Với mỗi `i`, **rút hết** thừa số `i`: `while num % i == 0: num //= i`; mỗi lần rút, đặt chữ số `i` vào hàng `mul` bằng `ans = mul * i + ans; mul *= 10` (`mul = 1, 10, 100, ...`). Chữ số rút **trước** (lớn hơn) nằm ở hàng **thấp** hơn, chữ số rút **sau** (nhỏ hơn) nằm ở hàng **cao** hơn, nên các chữ số xếp **tăng dần từ trái sang phải**. Ví dụ `48`: rút `8` → `ans = 8`; rút `6` → `ans = 10·6 + 8 = 68`.
3. Sau vòng lặp: nếu `num > 1` thì còn thừa số nguyên tố `> 7` → không có đáp án. Nếu `ans > 2^31 - 1` → tràn 32-bit. Cả hai trả `0`.

In [13]:
def smallest_greedy(num):
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans       # đặt chữ số i vào hàng mul
            mul *= 10
    return ans if num < 2 and ans <= INT_MAX else 0


for num in (48, 15, 1, 13, 100, 360, 2 ** 30):
    print(f"num={num:>11d} -> {smallest_greedy(num)}")
assert smallest_greedy(48) == 68 and smallest_greedy(15) == 35 and smallest_greedy(1) == 1

num=         48 -> 68
num=         15 -> 35
num=          1 -> 1
num=         13 -> 0
num=        100 -> 455
num=        360 -> 589
num= 1073741824 -> 0


In [14]:
# Code nộp LeetCode cho Cách 3 (đáp án chính)
class SolutionGreedy:
    def smallestFactorization(self, num: int) -> int:
        if num < 2:
            return num
        ans, mul = 0, 1
        for i in range(9, 1, -1):
            while num % i == 0:
                num //= i
                ans = mul * i + ans
                mul *= 10
        return ans if num < 2 and ans <= 2 ** 31 - 1 else 0

**Độ phức tạp:** vòng `for` 8 chữ số; mỗi `while` chạy tối đa `log2(num)` lần: tổng `O(log num)` thời gian, `O(1)` bộ nhớ.

### Cách 4 — Đếm số mũ nguyên tố rồi ghép chữ số

**Ý tưởng khác:** thay vì "chia thử", ta phân tích `num = 2^a · 3^b · 5^c · 7^d` (nếu còn dư → `0`). Rồi **ghép** các thừa số thành chữ số, **gói** nguyên tố vào chữ số lớn nhất có thể:

- `7^d` → `d` chữ số `7`; `5^c` → `c` chữ số `5` (nguyên tố lớn không ghép được với gì: `5·2 = 10` không phải chữ số).
- `3^b` → `b // 2` chữ số `9`, dư `b % 2` thừa số `3`.
- `2^a` → `a // 3` chữ số `8`, dư `a % 3` thừa số `2`.
- Ghép phần dư `(3^r3, 2^r2)` với `r3 ∈ {0,1}`, `r2 ∈ {0,1,2}` bằng bảng 6 trường hợp:

| `(r3, r2)` | chữ số ghép | giá trị |
|---|---|---|
| (0, 0) | — | 1 |
| (0, 1) | `2` | 2 |
| (0, 2) | `4` | 4 |
| (1, 0) | `3` | 3 |
| (1, 1) | `6` | 6 |
| (1, 2) | `2, 6` (= 2·6 = 12) | 12 |

Trường hợp `(1, 2)` (tức `12`) có hai cách: `{2, 6}` hoặc `{3, 4}`; cả hai đều hai chữ số nhưng `26 < 34` nên chọn `{2, 6}`. Cuối cùng **sắp xếp tăng dần** các chữ số.

In [15]:
TAIL = {(0, 0): [], (0, 1): [2], (0, 2): [4], (1, 0): [3], (1, 1): [6], (1, 2): [2, 6]}


def smallest_primes(num):
    if num == 1:
        return 1
    rest, (a, b, c, d) = small_prime_part(num)           # mũ của 2, 3, 5, 7
    if rest != 1:
        return 0
    digits = [9] * (b // 2) + [8] * (a // 3) + [7] * d + [5] * c + TAIL[(b % 2, a % 3)]
    ans = int("".join(map(str, sorted(digits))))
    return ans if ans <= INT_MAX else 0


for num in (48, 15, 1, 13, 100, 360, 12, 24, 2 ** 30):
    print(f"num={num:>11d} -> {smallest_primes(num)}")
assert smallest_primes(48) == 68 and smallest_primes(15) == 35 and smallest_primes(12) == 26

num=         48 -> 68
num=         15 -> 35
num=          1 -> 1
num=         13 -> 0
num=        100 -> 455
num=        360 -> 589
num=         12 -> 26
num=         24 -> 38
num= 1073741824 -> 0


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionPrimes:
    def smallestFactorization(self, num: int) -> int:
        if num == 1:
            return 1
        exps = []
        for p in (2, 3, 5, 7):
            e = 0
            while num % p == 0:
                num //= p
                e += 1
            exps.append(e)
        if num != 1:                      # còn ước nguyên tố > 7
            return 0
        a, b, c, d = exps
        tail = {(0, 0): [], (0, 1): [2], (0, 2): [4], (1, 0): [3], (1, 1): [6], (1, 2): [2, 6]}
        digits = [9] * (b // 2) + [8] * (a // 3) + [7] * d + [5] * c + tail[(b % 2, a % 3)]
        ans = int("".join(map(str, sorted(digits))))
        return ans if ans <= 2 ** 31 - 1 else 0

**Độ phức tạp:** `O(log num)` thời gian (đếm số mũ), `O(log num)` bộ nhớ để lưu danh sách chữ số (tối đa vài chục chữ số). Cùng bậc với Cách 3 nhưng *giải thích rõ vì sao* chọn các chữ số như vậy (bảng 6 trường hợp).

## 8. TRACE từng bước

Trace Cách 3 (tham lam) với `num = 48`:

| `i` | `num` trước | `num % i` | hành động | `num` sau | `ans` | `mul` |
|---|---|---|---|---|---|---|
| 9 | 48 | 3 | không chia hết | 48 | 0 | 1 |
| 8 | 48 | 0 | `num = 6`, `ans = 1·8 + 0 = 8`, `mul = 10` | 6 | 8 | 10 |
| 8 | 6 | 6 | dừng `while` | 6 | 8 | 10 |
| 7 | 6 | 6 | không chia hết | 6 | 8 | 10 |
| 6 | 6 | 0 | `num = 1`, `ans = 10·6 + 8 = 68`, `mul = 100` | 1 | 68 | 100 |
| 5..2 | 1 | — | không chia hết | 1 | 68 | 100 |

Cuối: `num = 1 < 2` và `68 <= 2^31 - 1` → trả `68`. Hàm trace dưới đây in từng vòng, gồm cả ví dụ có số dư (`13`, `26`) và `num = 12` (cặp `2, 6`).

In [17]:
def trace_greedy(num):
    print(f"=== num = {num} ===")
    if num < 2:
        print("num < 2 -> trả", num)
        return num
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        before = num
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            print(f"  i={i}: {before} chia hết cho {i} -> num={num}, ans={ans}, mul={mul}*10")
            mul *= 10
        else:
            print(f"  i={i}: kết thúc rút ({num} % {i} = {num % i}), ans={ans}")
    ok = num < 2 and ans <= INT_MAX
    print(f"  cuối: num={num}, ans={ans} -> {'trả ' + str(ans) if ok else 'trả 0 (' + ('còn dư > 1' if num >= 2 else 'tràn 32-bit') + ')'}")
    return ans if ok else 0


assert trace_greedy(48) == 68
assert trace_greedy(12) == 26
assert trace_greedy(26) == 0
assert trace_greedy(1) == 1

=== num = 48 ===
  i=9: kết thúc rút (48 % 9 = 3), ans=0
  i=8: 48 chia hết cho 8 -> num=6, ans=8, mul=1*10
  i=8: kết thúc rút (6 % 8 = 6), ans=8
  i=7: kết thúc rút (6 % 7 = 6), ans=8
  i=6: 6 chia hết cho 6 -> num=1, ans=68, mul=10*10
  i=6: kết thúc rút (1 % 6 = 1), ans=68
  i=5: kết thúc rút (1 % 5 = 1), ans=68
  i=4: kết thúc rút (1 % 4 = 1), ans=68
  i=3: kết thúc rút (1 % 3 = 1), ans=68
  i=2: kết thúc rút (1 % 2 = 1), ans=68
  cuối: num=1, ans=68 -> trả 68
=== num = 12 ===
  i=9: kết thúc rút (12 % 9 = 3), ans=0
  i=8: kết thúc rút (12 % 8 = 4), ans=0
  i=7: kết thúc rút (12 % 7 = 5), ans=0
  i=6: 12 chia hết cho 6 -> num=2, ans=6, mul=1*10
  i=6: kết thúc rút (2 % 6 = 2), ans=6
  i=5: kết thúc rút (2 % 5 = 2), ans=6
  i=4: kết thúc rút (2 % 4 = 2), ans=6
  i=3: kết thúc rút (2 % 3 = 2), ans=6
  i=2: 2 chia hết cho 2 -> num=1, ans=26, mul=10*10
  i=2: kết thúc rút (1 % 2 = 1), ans=26
  cuối: num=1, ans=26 -> trả 26
=== num = 26 ===
  i=9: kết thúc rút (26 % 9 = 8), ans=0
  i=8

**Trace Cách 4** cho `num = 360 = 2³ · 3² · 5`: số mũ `(a, b, c, d) = (3, 2, 1, 0)`; `b // 2 = 1` chữ số `9`; `a // 3 = 1` chữ số `8`; `c = 1` chữ số `5`; phần dư `(b % 2, a % 3) = (0, 0)` rỗng. Chữ số `{9, 8, 5}` sắp tăng dần: `589`.

In [18]:
def trace_primes(num):
    print(f"=== num = {num} ===")
    rest, (a, b, c, d) = small_prime_part(num)
    print(f"  mũ của (2,3,5,7) = {(a, b, c, d)}, số dư = {rest}")
    if num == 1:
        print("  num = 1 -> 1")
        return 1
    if rest != 1:
        print("  còn ước nguyên tố > 7 -> 0")
        return 0
    parts = {"9 (từ 3^2)": [9] * (b // 2), "8 (từ 2^3)": [8] * (a // 3), "7": [7] * d, "5": [5] * c, "phần dư": TAIL[(b % 2, a % 3)]}
    for name, ds in parts.items():
        print(f"  {name:12s}: {ds}")
    digits = sorted(sum(parts.values(), []))
    ans = int("".join(map(str, digits)))
    print(f"  gộp và sắp tăng dần: {digits} -> {ans}")
    return ans


assert trace_primes(360) == 589
assert trace_primes(48) == 68
assert trace_primes(2 ** 4 * 3 ** 3) == smallest_greedy(432)

=== num = 360 ===
  mũ của (2,3,5,7) = (3, 2, 1, 0), số dư = 1
  9 (từ 3^2)  : [9]
  8 (từ 2^3)  : [8]
  7           : []
  5           : [5]
  phần dư     : []
  gộp và sắp tăng dần: [5, 8, 9] -> 589
=== num = 48 ===
  mũ của (2,3,5,7) = (4, 1, 0, 0), số dư = 1
  9 (từ 3^2)  : []
  8 (từ 2^3)  : [8]
  7           : []
  5           : []
  phần dư     : [6]
  gộp và sắp tăng dần: [6, 8] -> 68
=== num = 432 ===
  mũ của (2,3,5,7) = (4, 3, 0, 0), số dư = 1
  9 (từ 3^2)  : [9]
  8 (từ 2^3)  : [8]
  7           : []
  5           : []
  phần dư     : [6]
  gộp và sắp tăng dần: [6, 8, 9] -> 689


## 9. Code nộp LeetCode

Chọn **tham lam chia cho `9..2`** vì ngắn, nhanh `O(log num)`. Giữ nguyên tên `class Solution`, `smallestFactorization(self, num)`:

In [19]:
class Solution:
    def smallestFactorization(self, num: int) -> int:
        if num < 2:                 # num = 1: chính 1 (tích chữ số của 1 là 1)
            return num
        ans, mul = 0, 1             # mul = giá trị hàng sẽ đặt chữ số tiếp theo
        for i in range(9, 1, -1):   # thử chữ số lớn trước: 9, 8, ..., 2
            while num % i == 0:
                num //= i
                ans = mul * i + ans # đặt chữ số i vào hàng mul
                mul *= 10
        # num còn > 1: có ước nguyên tố > 7; ans > 2^31-1: tràn số nguyên 32-bit
        return ans if num < 2 and ans <= 2 ** 31 - 1 else 0


for n in (48, 15, 1, 13):
    print(n, "->", Solution().smallestFactorization(n))
assert [Solution().smallestFactorization(n) for n in (48, 15, 1, 13)] == [68, 35, 1, 0]

48 -> 68
15 -> 35
1 -> 1
13 -> 0


## 10. Vì sao đúng

**Khung chứng minh.** Mọi đáp án là một **đa tập chữ số `D ⊆ {2..9}`** có tích `num` (trừ `num = 1`) xếp tăng dần. Ta chứng minh tham lam chọn đúng `D` tối ưu.

**(a) Số nhỏ ↔ đa tập "ít chữ số trước, rồi từ điển".** Đã nói ở 3.5.

**(b) Trong đáp án tối ưu, mọi cặp chữ số có tích `≥ 10`.** Nếu có hai chữ số `x, y` với `x·y ≤ 9` thì thay chúng bằng **một** chữ số `x·y` (vẫn trong `2..9`) → ít chữ số hơn → số nhỏ hơn: mâu thuẫn. Hệ quả:

- không có hai chữ số `3` (`3·3 = 9` → gộp thành `9`), không có `2` cùng `3` (`2·3 = 6` → gộp thành `6`), không có hai chữ số `2` (→ `4`), không có `2` cùng `4` (→ `8`), v.v.
- Do đó các chữ số nhỏ `2, 3, 4` chỉ xuất hiện rất hạn chế (ví dụ `{2, 6}`, `{3, 4}` hoặc riêng lẻ).

**(c) Các thừa số nguyên tố.** `5` và `7` chỉ có trong chữ số `5` và `7` → bắt buộc, mỗi cái đúng bằng số mũ. Phần `2^a·3^b` dùng các chữ số `{2,3,4,6,8,9}`. Từ (b): số chữ số `9` là `b // 2` (nếu `b ≥ 2` mà không dùng `9` thì hai thừa số `3` nằm ở các chữ số như `3, 3`, `3, 6` hoặc `6, 6`, và các cặp này luôn đổi được thành ít chữ số hơn); số chữ số `8` là `a // 3` với lý lẽ tương tự; phần dư `(b % 2, a % 3)` chỉ có 6 trường hợp, mỗi trường hợp có đáp án tối ưu hiển nhiên (bảng ở Cách 4, trường hợp `(1,2)` chọn `{2,6}` thay vì `{3,4}` vì `26 < 34`).

**(d) Cách 3 cho đúng đa tập đó.** Tham lam chia `9, 8, 7, 6, 5, 4, 3, 2`: rút hết `9`: `b // 2` cái (nếu `b` lẻ còn một `3`); rút hết `8`: `a // 3` cái; `7` và `5` rút hết; còn lại thừa số `2^(a%3)·3^(b%2)`; với `6`: nếu còn cả `2` và `3` thì rút một `6`; sau đó `4`, `3`, `2` rút nốt. Trường hợp `(1,2)`: `6` rồi `2` → `{2,6}` ✓; `(0,2)` → `4` ✓; `(1,1)` → `6` ✓; v.v. Trùng bảng của Cách 4.

**(e) Điều kiện trả `0`.** `num > 1` sau vòng lặp ⇒ có ước nguyên tố `> 7` ⇒ không đa tập nào. Và nếu số kết quả `> 2^31 - 1` thì theo đề trả `0`. Mọi đáp án 10 chữ số đều bắt đầu bằng chữ số `≥ 2` và chữ số thứ hai `≥ 2`, tức `≥ 22xxxxxxxx > 2147483647`; nên **mọi đáp án 10 chữ số trở lên đều tràn**, chỉ các đáp án tối đa 9 chữ số mới hợp lệ. Ta kiểm chứng điều này bằng máy ở mục 12.

Ta tự kiểm chứng toàn bộ bằng **hai oracle độc lập**: (1) bảng "tích chữ số" đầy đủ cho mọi `x < 10^6` (vét cạn), (2) quy hoạch động trên các ước (chọn chữ số đầu tiên, so sánh `(độ dài, chuỗi)`).

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử `x = 1, 2, ...` | `O(đáp án)` — tới `~10^9`; vô hạn nếu không có đáp án | `O(1)` | chỉ làm oracle (có `limit`) |
| 2. Quay lui liệt kê | `O(số cách tách)` (vài nghìn) | `O(log num)` (độ sâu) | đúng nhưng thừa |
| 3. Tham lam `9..2` | `O(log num)` | `O(1)` | **đáp án chính** |
| 4. Mũ nguyên tố + ghép | `O(log num)` | `O(log num)` | giải thích rõ vì sao tham lam đúng |

`log num` ở đây là cỡ `log2(2^31) = 31` bước chia: mỗi `while` đều chia `num` ít nhất một nửa. Đo thực tế (đều tức thời):

In [20]:
def bench(fn, nums, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        for n in nums:
            fn(n)
        best = min(best, time.perf_counter() - t0)
    return best


rng = random.Random(9)
# 20000 số "trơn" (chỉ chứa 2,3,5,7) lớn: ước lượng thời gian từng cách trên 20000 lần gọi
def random_smooth(rng, limit=INT_MAX):
    while True:
        e = [rng.randint(0, 30), rng.randint(0, 19), rng.randint(0, 13), rng.randint(0, 11)]
        n = 2 ** e[0] * 3 ** e[1] * 5 ** e[2] * 7 ** e[3]
        if 1 <= n <= limit:
            return n


nums = [random_smooth(rng) for _ in range(20000)]
for name, fn in [("greedy", smallest_greedy), ("primes", smallest_primes), ("backtrack", smallest_backtrack)]:
    print(f"{name:10s} {bench(fn, nums[:2000 if name == 'backtrack' else 20000]) * 1000:9.2f} ms",
          "(2000 lần)" if name == "backtrack" else "(20000 lần)")
t0 = time.perf_counter()
smallest_brute(48)
print(f"brute 48 mất {(time.perf_counter() - t0) * 1000:.3f} ms; brute 13 (không đáp án, limit=10^6) mất {bench(lambda n: smallest_brute(n), [13], 1) * 1000:.0f} ms")

greedy         25.06 ms (20000 lần)
primes         56.64 ms (20000 lần)


backtrack    1775.71 ms (2000 lần)


brute 48 mất 0.164 ms; brute 13 (không đáp án, limit=10^6) mất 408 ms


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ: cài đặt sai phải bị bắt, (D) **đối chiếu với hai oracle độc lập** trên mọi `num` nhỏ, (E) ngẫu nhiên có seed đúng ràng buộc, (F) biên 32-bit.

### A. Ví dụ của đề

In [21]:
IMPLS = {
    "brute": lambda n: smallest_brute(n),
    "backtrack": smallest_backtrack,
    "greedy": smallest_greedy,
    "primes": smallest_primes,
    "Solution": lambda n: Solution().smallestFactorization(n),
    "SolutionBacktrack": lambda n: SolutionBacktrack().smallestFactorization(n),
    "SolutionGreedy": lambda n: SolutionGreedy().smallestFactorization(n),
    "SolutionPrimes": lambda n: SolutionPrimes().smallestFactorization(n),
}
FAST = [k for k in IMPLS if k != "brute"]       # brute chỉ dùng cho num nhỏ


def check(num, want, names=None):
    for name in (names or FAST):
        got = IMPLS[name](num)
        assert got == want, (name, num, got, want)


for num, want in [(48, 68), (15, 35)]:
    check(num, want)
    assert IMPLS["brute"](num) == want
print("ví dụ 1 và 2: mọi cài đặt đúng")

ví dụ 1 và 2: mọi cài đặt đúng


### B. Edge case

- `num = 1` → `1`; `num = 2..9` → chính nó (một chữ số).
- `num = 10` → `25`; `num = 11, 13, 17, 19, 22, 26` → `0` (có ước nguyên tố `> 7`).
- `num = 12` → `26` (không phải `34`); `num = 16` → `28` (`2·8`); `num = 36` → `49`; `num = 100` → `455`.
- Số nguyên tố lớn như `2^31 − 1` → `0`.
- `num` có ước `7` và `5`, lũy thừa lớn.

In [22]:
edge = [
    (1, 1), (2, 2), (3, 3), (4, 4), (5, 5), (6, 6), (7, 7), (8, 8), (9, 9),
    (10, 25), (12, 26), (16, 28), (18, 29), (20, 45), (24, 38), (36, 49), (100, 455), (360, 589),
    (11, 0), (13, 0), (17, 0), (19, 0), (22, 0), (26, 0), (121, 0), (INT_MAX, 0),
]
for num, want in edge:
    check(num, want)
    if num <= 1000:
        assert IMPLS["brute"](num) == want, num
print(len(edge), "edge case: mọi cài đặt đúng (brute đối chiếu cho num <= 1000)")

26 edge case: mọi cài đặt đúng (brute đối chiếu cho num <= 1000)


### C. Phản ví dụ: cài đặt sai phải bị bắt

Ta viết các bản sai (chi tiết ở mục 13) và chắc chắn chúng bị bắt bởi oracle:

In [23]:
def bug_ascending(num):                 # duyệt chữ số từ 2 lên 9
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(2, 10):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if num < 2 and ans <= INT_MAX else 0


def bug_order(num):                     # ghép chữ số theo thứ tự lớn trước (ans*10 + i)
    if num < 2:
        return num
    ans = 0
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = ans * 10 + i
    return ans if num < 2 and ans <= INT_MAX else 0


def bug_no_remainder_check(num):        # quên kiểm tra num còn > 1
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if ans <= INT_MAX else 0


def bug_num_one(num):                   # quên num = 1
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if num < 2 and ans <= INT_MAX else 0


def bug_no_overflow(num):               # quên giới hạn 32-bit
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if num < 2 else 0


def bug_range_end(num):                 # range(9, 2, -1): bỏ sót chữ số 2
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(9, 2, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if num < 2 and ans <= INT_MAX else 0


smooth_tests = [2 ** a * 3 ** b * 5 ** c * 7 ** d for a in range(0, 32, 3) for b in range(0, 6) for c in range(0, 4) for d in range(0, 3)]
tests = [n for n in range(1, 400)] + [n for n in smooth_tests if n <= INT_MAX]
for bug in (bug_ascending, bug_order, bug_no_remainder_check, bug_num_one, bug_no_overflow, bug_range_end):
    bad = next(n for n in tests if bug(n) != smallest_primes(n))
    print(f"{bug.__name__:24s} sai tại num={bad}: sai {bug(bad)}, đúng {smallest_primes(bad)}")

bug_ascending            sai tại num=4: sai 22, đúng 4
bug_order                sai tại num=10: sai 52, đúng 25
bug_no_remainder_check   sai tại num=22: sai 2, đúng 0
bug_num_one              sai tại num=1: sai 0, đúng 1
bug_no_overflow          sai tại num=95256000: sai 3555778899, đúng 0
bug_range_end            sai tại num=2: sai 0, đúng 2


### D. Đối chiếu với hai oracle độc lập trên mọi `num` nhỏ

**Oracle 1 — bảng vét cạn:** với mọi `x < 10^6` tính tích chữ số, giữ `x` nhỏ nhất cho từng tích. Mọi `num ≤ 5000` mà đáp án tồn tại đều có đáp án `< 10^6` (đáp án dài nhất cho `num ≤ 5000` là `5^5 = 3125 → 55555`). Vậy `table.get(num, 0)` là đáp án đúng cho mọi `num ≤ 5000`.

**Oracle 2 — quy hoạch động:** `best(n)` là chuỗi chữ số nhỏ nhất (theo `(độ dài, từ điển)`) có tích `n`: chọn chữ số đầu `k | n` rồi ghép `k + best(n / k)`. Độc lập với tham lam.

In [24]:
# Oracle 1: bảng tích chữ số cho x < 10^6
LIM = 10 ** 6
prod = [0] * LIM
prod[0] = 0
first_x = {}
for x in range(1, LIM):
    prod[x] = prod[x // 10] * (x % 10) if x >= 10 else x
    if x < 10:
        prod[x] = x
    p = prod[x]
    if p not in first_x:
        first_x[p] = x
print("bảng vét cạn:", len(first_x), "giá trị tích khác nhau; ví dụ", first_x[48], first_x[15], first_x[1])
assert first_x[48] == 68 and first_x[15] == 35 and first_x[1] == 1


# Oracle 2: quy hoạch động theo ước số
@lru_cache(maxsize=None)
def dp_best(n):
    """Chuỗi chữ số nhỏ nhất (theo độ dài rồi từ điển) có tích n; None nếu không có. n == 1 -> ''."""
    if n == 1:
        return ""
    best = None
    for k in range(2, 10):
        if n % k == 0:
            rest = dp_best(n // k)
            if rest is None:
                continue
            cand = str(k) + rest
            if best is None or (len(cand), cand) < (len(best), best):
                best = cand
    return best


def oracle_dp(num):
    if num == 1:
        return 1
    s = dp_best(num)
    return 0 if s is None or int(s) > INT_MAX else int(s)


assert oracle_dp(48) == 68 and oracle_dp(15) == 35 and oracle_dp(13) == 0 and oracle_dp(1) == 1

# đối chiếu: mọi num 1..5000
for num in range(1, 5001):
    want = first_x.get(num, 0)
    assert oracle_dp(num) == want, ("dp vs bảng", num, oracle_dp(num), want)
    check(num, want, ["greedy", "primes", "backtrack", "Solution", "SolutionGreedy", "SolutionPrimes", "SolutionBacktrack"])
print("num = 1..5000: bảng vét cạn = DP = 7 cài đặt")

# thêm: mọi num 5001..60000 so với DP (oracle độc lập)
for num in range(5001, 60001):
    want = oracle_dp(num)
    assert smallest_greedy(num) == want and smallest_primes(num) == want, num
print("num = 5001..60000: greedy = primes = DP")

bảng vét cạn: 785 giá trị tích khác nhau; ví dụ 68 35 1
num = 1..5000: bảng vét cạn = DP = 7 cài đặt
num = 5001..60000: greedy = primes = DP


### E. Ngẫu nhiên có seed, đúng ràng buộc

Có hai loại input: (1) số **trơn** (chỉ chứa `2, 3, 5, 7`) sinh bằng cách nhân các chữ số ngẫu nhiên — có đáp án; (2) số hoàn toàn ngẫu nhiên trong `[1, 2^31 − 1]` — hầu như đều chứa ước nguyên tố lớn nên đáp án `0`. So với oracle DP.

In [25]:
rng = random.Random(20240625)
n_smooth = n_rand = 0
for _ in range(4000):
    # loại 1: tích của vài chữ số ngẫu nhiên 2..9 (<= 2^31-1)
    n = 1
    for _ in range(rng.randint(1, 14)):
        d = rng.randint(2, 9)
        if n * d <= INT_MAX:
            n *= d
    want = oracle_dp(n)
    check(n, want)
    n_smooth += 1
    # loại 2: số ngẫu nhiên bất kỳ
    m = rng.randint(1, INT_MAX)
    want = oracle_dp(m)
    check(m, want)
    n_rand += 1
print(n_smooth, "số trơn +", n_rand, "số ngẫu nhiên: mọi cài đặt trùng DP")

# số ngẫu nhiên có đáp án khác 0 hiếm, thống kê:
zeros = sum(1 for _ in range(2000) if oracle_dp(rng.randint(1, INT_MAX)) == 0)
print("trong 2000 số ngẫu nhiên có", zeros, "số cho đáp án 0")

# số mũ ngẫu nhiên: 2^a 3^b 5^c 7^d, tất cả số trơn <= INT_MAX (kiểm chứng toàn bộ tập)
smooth_all = []
a = 1
for e2 in range(0, 32):
    for e3 in range(0, 20):
        for e5 in range(0, 14):
            for e7 in range(0, 12):
                v = 2 ** e2 * 3 ** e3 * 5 ** e5 * 7 ** e7
                if v <= INT_MAX:
                    smooth_all.append(v)
print("số trơn <= INT_MAX:", len(smooth_all))
for v in smooth_all:
    want = oracle_dp(v)
    assert smallest_greedy(v) == want and smallest_primes(v) == want, v
print("toàn bộ", len(smooth_all), "số trơn: greedy = primes = DP")

4000 số trơn + 4000 số ngẫu nhiên: mọi cài đặt trùng DP
trong 2000 số ngẫu nhiên có 2000 số cho đáp án 0
số trơn <= INT_MAX: 5911
toàn bộ 5911 số trơn: greedy = primes = DP


### F. Biên 32-bit

Ta kiểm chứng lời khẳng định ở mục 10: **mọi đáp án có từ 10 chữ số trở lên đều tràn 32-bit**, trên **toàn bộ** số trơn `≤ 2^31 − 1`; và liệt kê các ví dụ sát biên.

In [26]:
def raw_answer(num):
    """Đáp án chưa cắt giới hạn 32-bit (None nếu không tách được)."""
    s = dp_best(num) if num > 1 else "1"
    return None if s is None else int(s)


max_len_ok = 0
for v in smooth_all:
    r = raw_answer(v)
    if r is None:
        continue
    if len(str(r)) >= 10:
        assert r > INT_MAX, (v, r)
    else:
        assert r <= INT_MAX, (v, r)
        max_len_ok = max(max_len_ok, len(str(r)))
print("mọi đáp án >= 10 chữ số đều > INT_MAX; đáp án dài nhất còn hợp lệ có", max_len_ok, "chữ số")

for num, desc in [(7 ** 9, "7^9"), (7 ** 10, "7^10"), (5 ** 12, "5^12"), (5 ** 13, "5^13"), (2 ** 27, "2^27 = 8^9"), (2 ** 30, "2^30 = 8^10")]:
    print(f"{desc:12s} num={num:>11d} -> thô {raw_answer(num)} -> nộp {Solution().smallestFactorization(num)}")
assert Solution().smallestFactorization(7 ** 9) == int("7" * 9)
assert Solution().smallestFactorization(7 ** 10) == 0
assert Solution().smallestFactorization(2 ** 27) == int("8" * 9)
assert Solution().smallestFactorization(2 ** 30) == 0

mọi đáp án >= 10 chữ số đều > INT_MAX; đáp án dài nhất còn hợp lệ có 9 chữ số
7^9          num=   40353607 -> thô 777777777 -> nộp 777777777
7^10         num=  282475249 -> thô 7777777777 -> nộp 0
5^12         num=  244140625 -> thô 555555555555 -> nộp 0
5^13         num= 1220703125 -> thô 5555555555555 -> nộp 0
2^27 = 8^9   num=  134217728 -> thô 888888888 -> nộp 888888888
2^30 = 8^10  num= 1073741824 -> thô 8888888888 -> nộp 0


## 13. Các lỗi người mới hay gặp

### Lỗi 1: duyệt chữ số từ 2 lên 9

Dễ nghĩ "thử ước nhỏ trước". Nhưng chữ số nhỏ chia được nhiều lần nên cho **nhiều chữ số hơn**, nên số lớn hơn.

In [27]:
# CODE SAI
def sai1(num):
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(2, 10):                # SAI: từ 2 lên 9
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if num < 2 and ans <= INT_MAX else 0


for n in (48, 12, 100):
    print(f"num={n}: sai1 = {sai1(n)}, đúng = {Solution().smallestFactorization(n)}")
assert sai1(48) != 68

num=48: sai1 = 32222, đúng = 68
num=12: sai1 = 322, đúng = 26
num=100: sai1 = 5522, đúng = 455


**Vì sao sai:** `48`: chia cho `2` bốn lần rồi `3` ra `2223 ... → 22223`-dạng (5 chữ số). **Đúng:** từ chữ số lớn xuống nhỏ để ít chữ số nhất (mục 6).

### Lỗi 2: ghép chữ số sai thứ tự

Duyệt `9 → 2` và viết `ans = ans * 10 + i` đặt chữ số lớn ở **hàng cao** → số lớn (ra `86` thay vì `68`). Muốn chữ số nhỏ ở hàng cao, phải đặt chữ số vào hàng *thấp nhất còn trống* với `mul` tăng dần.

In [28]:
# CODE SAI
def sai2(num):
    if num < 2:
        return num
    ans = 0
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = ans * 10 + i             # SAI: chữ số lớn đứng đầu
    return ans if num < 2 and ans <= INT_MAX else 0


print("sai2(48) =", sai2(48), "(đúng 68)", "| sai2(15) =", sai2(15), "(đúng 35)")
assert sai2(48) == 86

sai2(48) = 86 (đúng 68) | sai2(15) = 53 (đúng 35)


**Vì sao sai:** chữ số lớn đứng ở hàng cao → số lớn. **Đúng:** `ans = mul * i + ans; mul *= 10` (chữ số đầu tiên rút được là **lớn nhất** nằm ở hàng thấp nhất), hoặc gom chữ số vào list rồi `sorted`.

### Lỗi 3: quên kiểm tra phần dư `num > 1`

Nếu `num` còn thừa ước nguyên tố `> 7` thì không có đáp án, nhưng `ans` đã chứa vài chữ số "dở dang". Nếu chỉ trả `ans` thì có đáp án giả.

In [29]:
# CODE SAI
def sai3(num):
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if ans <= INT_MAX else 0          # SAI: quên num < 2


for n in (13, 26, 22, 11 * 8):
    print(f"num={n}: sai3 = {sai3(n)}, đúng = {Solution().smallestFactorization(n)}")
assert sai3(26) == 2 and Solution().smallestFactorization(26) == 0

num=13: sai3 = 0, đúng = 0
num=26: sai3 = 2, đúng = 0
num=22: sai3 = 2, đúng = 0
num=88: sai3 = 8, đúng = 0


**Vì sao sai:** `26 = 2 · 13`: chữ số `2` rút được nhưng còn `13` thừa → vô nghiệm. **Đúng:** `return ans if num < 2 and ... else 0`.

### Lỗi 4: quên `num = 1`

Với `num = 1` không chữ số nào `2..9` chia hết, nên `ans = 0` và vòng lặp trả `0`. Nhưng đáp án đúng là `1` (số `1` có tích chữ số bằng `1`).

In [30]:
# CODE SAI
def sai4(num):
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if num < 2 and ans <= INT_MAX else 0


print("sai4(1) =", sai4(1), "| đúng =", Solution().smallestFactorization(1))
assert sai4(1) == 0 and Solution().smallestFactorization(1) == 1

sai4(1) = 0 | đúng = 1


**Vì sao sai:** trường hợp suy biến. **Đúng:** `if num < 2: return num` ở đầu.

### Lỗi 5: quên giới hạn 32-bit hoặc `range(9, 2, -1)`

Python không tràn số nên dễ quên so với `2^31 - 1`; và `range(9, 2, -1)` dừng ở `3`, bỏ chữ số `2`.

In [31]:
# CODE SAI (quên giới hạn 32-bit)
def sai5(num):
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if num < 2 else 0


print("num = 7^10:", "sai5 =", sai5(7 ** 10), "| đúng =", Solution().smallestFactorization(7 ** 10))


# CODE SAI (range kết thúc ở 3 -> bỏ chữ số 2)
def sai6(num):
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(9, 2, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if num < 2 and ans <= INT_MAX else 0


print("sai6(2) =", sai6(2), "(đúng 2) | sai6(10) =", sai6(10), "(đúng 25)")
assert sai5(7 ** 10) == int("7" * 10) and sai6(2) == 0

num = 7^10: sai5 = 7777777777 | đúng = 0
sai6(2) = 0 (đúng 2) | sai6(10) = 0 (đúng 25)


**Vì sao sai:** `7^10` cho `7777777777 > 2147483647` — đề yêu cầu `0`. `range(9, 2, -1)` không bao giờ thử `2`. **Đúng:** `range(9, 1, -1)` và so sánh `ans <= 2**31 - 1`.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tìm số nhỏ nhất/lớn nhất với tích (hoặc tổng) chữ số cho trước" → **tham lam theo chữ số**; luôn nhớ hai tiêu chí: **độ dài** rồi **thứ tự từ điển**.

**Mẹo:**

1. **Ít chữ số hơn thì nhỏ hơn:** cố gắng dùng chữ số lớn nhất có thể.
2. **Sắp xếp chữ số tăng dần** để số nhỏ nhất (giảm dần để số lớn nhất).
3. **Phân tích nguyên tố** `2, 3, 5, 7` giúp giải thích: `5` và `7` bắt buộc; `3` kết hợp thành `9`, `2` thành `8`, phần dư chỉ có 6 trường hợp.
4. **Đừng tin tham lam khi chưa kiểm chứng:** trong hệ cơ số khác tham lam có thể sai (xem mục 15). Hãy tự viết oracle DP/vét cạn cho input nhỏ.
5. **Giới hạn kiểu dữ liệu:** Python không tràn, nhưng đề có thể bắt "vừa 32-bit": tự so sánh với `2**31 - 1`.
6. **Hai thông điệp trả về:** "không tồn tại" và "không vừa" cùng trả `0`.

Các bài cùng họ: Smallest Integer Divisible by K (1015), Largest Number (179), Integer Break (343 — chia nhỏ để tối đa tích), Smallest String With A Given Numeric Value (1663).

## 15. Biến thể

1. **Tìm số LỚN nhất** có tích chữ số bằng `num`: nếu cho phép chữ số `1` thì có thể chèn `1` vô hạn nên không có số lớn nhất; vì vậy thường phải cố định số chữ số hoặc bỏ chữ số `1`.
2. **Cơ số khác `b`** (chữ số `2..b-1`): tham lam "lớn trước" **có thể sai**! (xem cell bên dưới).
3. **Tổng chữ số** thay vì tích: số nhỏ nhất có tổng chữ số `s` là `(s mod 9)` rồi `⌊s/9⌋` chữ số `9`.
4. **Đếm** số cách tách `num` thành chữ số (bài tập 5).
5. **Cho phép chữ số 1 và 0:** `1` chỉ làm dài thêm; `0` làm tích `0` (khi `num = 0` đáp án `10`).

Biến thể 2: so tham lam với DP theo cơ số `base`.

In [32]:
def dp_base(num, base):
    """Đáp án thật trong cơ số base: dãy chữ số (2..base-1) ngắn nhất, rồi nhỏ nhất theo từ điển; None nếu không có."""
    @lru_cache(maxsize=None)
    def f(n):
        if n == 1:
            return ()
        best = None
        for k in range(2, base):
            if n % k == 0:
                t = f(n // k)
                if t is None:
                    continue
                cand = (k,) + t
                if best is None or (len(cand), cand) < (len(best), best):
                    best = cand
        return best
    return f(num)


def greedy_base(num, base):
    ds = []
    for k in range(base - 1, 1, -1):
        while num % k == 0:
            num //= k
            ds.append(k)
    return tuple(sorted(ds)) if num == 1 else None


# cơ số 10: tham lam đúng
assert all(dp_base(n, 10) == greedy_base(n, 10) for n in range(2, 3000))
print("cơ số 10: tham lam = DP trên num 2..2999")
# cơ số 9 (chữ số 2..8): tham lam có thể sai
bad = [(n, dp_base(n, 9), greedy_base(n, 9)) for n in range(2, 3000) if dp_base(n, 9) != greedy_base(n, 9)]
print("cơ số 9: số num tham lam sai =", len(bad), "| ví dụ:", bad[:2])
n0, right, wrong = bad[0]
print(f"num={n0}: đáp án thật {right} (tích {right[0] * right[1] * right[2]}), tham lam ra {wrong}")

cơ số 10: tham lam = DP trên num 2..2999


cơ số 9: số num tham lam sai = 17 | ví dụ: [(72, (2, 6, 6), (3, 3, 8)), (216, (6, 6, 6), (3, 3, 3, 8))]
num=72: đáp án thật (2, 6, 6) (tích 72), tham lam ra (3, 3, 8)


In [33]:
# Biến thể 3: số nhỏ nhất có tổng chữ số = s
def smallest_with_digit_sum(s):
    k, r = divmod(s, 9)
    return int((str(r) if r else "") + "9" * k) if s > 0 else 0


for s in (1, 9, 10, 18, 25):
    x = smallest_with_digit_sum(s)
    print(f"s={s:2d} -> {x}", "| tổng chữ số =", sum(map(int, str(x))))
    assert sum(map(int, str(x))) == s
# đối chiếu vét cạn
for s in range(1, 40):
    brute = next(x for x in range(1, 10 ** 6) if sum(map(int, str(x))) == s)
    assert smallest_with_digit_sum(s) == brute, s
print("biến thể tổng chữ số: khớp vét cạn s = 1..39")

s= 1 -> 1 | tổng chữ số = 1
s= 9 -> 9 | tổng chữ số = 9
s=10 -> 19 | tổng chữ số = 10
s=18 -> 99 | tổng chữ số = 18
s=25 -> 799 | tổng chữ số = 25
biến thể tổng chữ số: khớp vét cạn s = 1..39


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử `x` tăng dần | Duyệt `x`, tính tích chữ số | `O(đáp án)` (tới `~10^9`, vô hạn nếu vô nghiệm) | `O(1)` | Làm oracle cho `num` nhỏ |
| 2. Quay lui | Liệt kê mọi đa tập chữ số có tích `num` | số cách tách (nhỏ) | `O(log num)` | Khi chưa tin tham lam; kiểm chứng ý tưởng |
| 3. Tham lam `9..2` | Chia cho chữ số lớn trước, đặt vào hàng thấp trước | `O(log num)` | `O(1)` | **Đáp án chính** |
| 4. Mũ nguyên tố + bảng ghép | Đếm mũ `2,3,5,7`; `9`,`8`, rồi 6 trường hợp dư | `O(log num)` | `O(log num)` | Giải thích/chứng minh vì sao tham lam đúng |
| (oracle) DP theo ước | `best(n) = min_k str(k) + best(n/k)` | số ước × 8 | số ước | Chỉ để kiểm tra |

Tất cả cho cùng đáp án trên `1..60000`, toàn bộ số trơn `≤ 2^31 − 1`, và 4000 + 4000 input ngẫu nhiên (mục 12).

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test ở cuối.

### Bài 1 (đoán output)

Đoán trước rồi mới chạy:

In [34]:
print(smallest_greedy(36), smallest_greedy(11), smallest_greedy(1), smallest_greedy(64))

49 0 1 88


### Bài 2 (tính tay)

- (a) Tính `smallestFactorization(360)`: phân tích `360 = 2^a · 3^b · 5`, rồi dùng bảng ở Cách 4.
- (b) Tính `smallestFactorization(100)` và `smallestFactorization(432)`.
- (c) Tại sao `num = 2^31 − 1` cho `0`?

Tự giải rồi kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện hàm tham lam:

In [35]:
def smallest_todo(num):
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        # TODO 1: while num chia hết cho i -> chia num cho i
        # TODO 2:     đặt chữ số i vào hàng mul của ans, rồi mul *= 10
        pass
    # TODO 3: trả ans nếu num đã về 1 và ans vừa 32-bit, ngược lại 0
    return None


# Khi làm xong, bỏ comment để kiểm tra:
# assert [smallest_todo(n) for n in (48, 15, 1, 13, 7 ** 10)] == [68, 35, 1, 0, 0]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết lớp `MySolution` bằng cách tham lam (hoặc Cách 4):

In [36]:
class MySolution:
    def smallestFactorization(self, num):
        pass            # TODO


# Khi làm xong:
# for n, w in [(48, 68), (15, 35), (1, 1), (11, 0), (360, 589)]:
#     assert MySolution().smallestFactorization(n) == w

### Bài 5 (mini challenge): đếm số cách tách

Viết `count_factorizations(num)` trả về **số đa tập** chữ số trong `{2..9}` có tích bằng `num` (không phân biệt thứ tự; `num = 1` có 1 cách: tập rỗng). Ví dụ `48` có `7` cách (xem bảng ở mục 4). Gợi ý: quy hoạch kiểu "đổi tiền": `f(n, k)` = số cách dùng chữ số `≤ k`.

In [37]:
def count_factorizations(num):
    pass            # TODO


# Khi làm xong:
# assert count_factorizations(48) == 7 and count_factorizations(1) == 1 and count_factorizations(13) == 0

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: phân tích ra `2, 3, 5, 7`.
- Mức 2: `36 = 4 · 9` hay `6 · 6`? Số nào ít chữ số hơn và nhỏ hơn?
- Mức 3: `36 → 9` rút trước (`36/9 = 4`) → `49`; `11` có ước `11 > 7` → `0`; `1 → 1`; `64 = 8 · 8` → `88`.

**Bài 2**

- Mức 1: `360 = 8 · 45 = 8 · 9 · 5`.
- Mức 2: chữ số `{9, 8, 5}`, xếp tăng dần.
- Mức 3: `589`. `100 = 2² · 5² → {4, 5, 5}` → `455`. `432 = 2^4 · 3^3`: `9` một lần (còn `3`), `8` một lần (còn `2`), dư `(1, 1)` → `6`: `{9, 8, 6}` → `689`. `2^31 − 1` là số nguyên tố (khác `2, 3, 5, 7`) nên `0`.

**Bài 3**

- Mức 1: giống vòng `while` ở mục 3.3.
- Mức 2: ba dòng trong `while`: `num //= i`, `ans = mul * i + ans`, `mul *= 10`.
- Mức 3: cuối: `return ans if num < 2 and ans <= 2**31 - 1 else 0`.

**Bài 4**

- Mức 1: `num < 2` trả `num`.
- Mức 2: vòng `for i in range(9, 1, -1)`.
- Mức 3: xem mục 9.

**Bài 5**

- Mức 1: đổi tiền: "mệnh giá" là chữ số `2..9`, nhưng phép "cộng" là **nhân**.
- Mức 2: `f(n, k) = f(n, k − 1) + f(n // k, k)` (nếu `k | n`), `f(1, k) = 1`, `f(n, 1) = 0` với `n > 1`.
- Mức 3: dùng `lru_cache`; hoặc đệ quy liệt kê như `all_factorizations` rồi đếm để kiểm tra.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert` để chạy kiểm tra.

### Bài 1

In [38]:
out = (smallest_greedy(36), smallest_greedy(11), smallest_greedy(1), smallest_greedy(64))
assert out == (49, 0, 1, 88)
print("Bài 1:", out)

Bài 1: (49, 0, 1, 88)


### Bài 2

(a) `360 = 2³ · 3² · 5` → `9` (từ `3²`), `8` (từ `2³`), `5` → `589`. (b) `100 = 2² · 5²` → `{4, 5, 5}` → `455`; `432 = 2⁴ · 3³` → `{9, 8, 6}` → `689`. (c) `2^31 − 1 = 2147483647` là số nguyên tố, lớn hơn `7`, nên không tách được.

In [39]:
S = Solution()
assert S.smallestFactorization(360) == 589
assert S.smallestFactorization(100) == 455
assert S.smallestFactorization(432) == 689
assert S.smallestFactorization(INT_MAX) == 0
rest, e = small_prime_part(INT_MAX)
assert rest == INT_MAX      # không có thừa số 2,3,5,7 nào
assert oracle_dp(432) == 689 and oracle_dp(100) == 455 and oracle_dp(360) == 589
print("Bài 2 OK")

Bài 2 OK


### Bài 3

In [40]:
def smallest_answer(num):
    if num < 2:
        return num
    ans, mul = 0, 1
    for i in range(9, 1, -1):
        while num % i == 0:
            num //= i
            ans = mul * i + ans
            mul *= 10
    return ans if num < 2 and ans <= 2 ** 31 - 1 else 0


assert [smallest_answer(n) for n in (48, 15, 1, 13, 7 ** 10)] == [68, 35, 1, 0, 0]
assert all(smallest_answer(n) == oracle_dp(n) for n in range(1, 3000))
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [41]:
class MySolutionAnswer:
    def smallestFactorization(self, num):
        if num == 1:
            return 1
        digits = []
        for d in range(9, 1, -1):
            while num % d == 0:
                num //= d
                digits.append(d)
        if num != 1:
            return 0
        ans = int("".join(map(str, sorted(digits))))
        return ans if ans <= 2 ** 31 - 1 else 0


for n, w in [(48, 68), (15, 35), (1, 1), (11, 0), (360, 589)]:
    assert MySolutionAnswer().smallestFactorization(n) == w
assert all(MySolutionAnswer().smallestFactorization(n) == oracle_dp(n) for n in range(1, 3000))
print("Bài 4 OK")

Bài 4 OK


### Bài 5

`f(n, k)`: số cách tách `n` bằng các chữ số trong `{2..k}`. Hoặc không dùng `k` (`f(n, k-1)`), hoặc dùng ít nhất một `k` (`f(n/k, k)`).

In [42]:
def count_factorizations_answer(num):
    @lru_cache(maxsize=None)
    def f(n, k):
        if n == 1:
            return 1
        if k == 1:
            return 0
        ways = f(n, k - 1)
        if n % k == 0:
            ways += f(n // k, k)
        return ways

    return f(num, 9)


assert count_factorizations_answer(48) == 7 and count_factorizations_answer(1) == 1 and count_factorizations_answer(13) == 0
# đối chiếu với liệt kê quay lui ở mục 4
for n in list(range(1, 500)) + [360, 1024, 2 ** 15 * 3 ** 8, 7 ** 5 * 5 ** 3 * 2 ** 6]:
    assert count_factorizations_answer(n) == count_all(n), n
print("Bài 5 OK: số cách tách của 48 =", count_factorizations_answer(48), "| của 360 =", count_factorizations_answer(360))

Bài 5 OK: số cách tách của 48 = 7 | của 360 = 9


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (số nhỏ nhất có tích chữ số = `num`):**

```python
class Solution:
    def smallestFactorization(self, num: int) -> int:
        if num < 2:
            return num                     # num = 1 -> 1
        ans, mul = 0, 1
        for i in range(9, 1, -1):          # chữ số lớn trước
            while num % i == 0:
                num //= i
                ans = mul * i + ans        # chữ số rút trước nằm ở hàng thấp
                mul *= 10
        return ans if num < 2 and ans <= 2 ** 31 - 1 else 0
```

**Checklist trước khi submit:**

- [ ] `num = 1` → `1` (đã xử lý riêng)?
- [ ] Duyệt chữ số **từ 9 xuống 2** (`range(9, 1, -1)`), không bỏ chữ số `2`?
- [ ] Ghép chữ số đúng thứ tự: chữ số **nhỏ ở hàng cao** (`ans = mul * i + ans`)?
- [ ] Kiểm tra `num < 2` (không còn ước nguyên tố `> 7`) trước khi trả đáp án?
- [ ] So sánh `ans <= 2**31 - 1` (đáp án 10 chữ số trở lên luôn tràn)?
- [ ] Đã thử `48 → 68`, `15 → 35`, `12 → 26`, `1 → 1`, `13 → 0`, `7^10 → 0`?
- [ ] Đúng tên `class Solution`, `smallestFactorization(self, num)`, không in debug.